In [1]:
import pandas as pd

df = pd.read_json("processed_df.jsonl", lines=True)


In [2]:
df.columns

Index(['text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches', 'entity_type'],
      dtype='object')

In [3]:
df.loc[1, 'uuid']

'0011edeb-0524-4d8f-be9b-d618b619b7e6'

# We focus on annotating 40 articles from the 19k regex-filtered dataset. We compute simpledorff score

In [4]:
rel_sample = df.sample(n=40, random_state = 5)
rel_export = rel_sample[['uuid', 'title', 'maintext']]

rel_export.to_csv("sample_40_for_relevance.csv", index=False)

In [5]:
import pandas as pd

df = pd.read_csv("combined_relevance_annotations_v1.csv")
df.head()



,uuid,title,maintext,jer_relevant,julia_relevant
0,7b195e77-ee84-490a-b7ce-9863580541dc,747 cargo jet's landing gear snaps off during ...,NaN,n,n
1,c61a6448-e74f-4e5e-9853-3dcd52268c24,Haitian Americans explain why they're staying ...,A dramatic emergency landing Sunday caught on ...,n,y
2,1349f4df-ab11-47fa-a80b-059b49199f9a,Jewish Man Taunted While Describing Antisemiti...,Footage of a Jewish man sharing his experience...,y,y
3,8d5cb78d-cd05-4b4e-8b78-0550a5639be4,"Tesla, GM and Ford are feeling the chill of th...",Share on email (opens in new window)\nShare on...,n,y
4,e32409d3-9676-42b8-ac59-2b1f5d1e3ab8,Southwest Airlines’ Pilots Have Had Enough. Wh...,"DENVER, CO - DECEMBER 28: Pristine Floyde sear...",y,y


In [6]:
df = df.replace({
    "y": 1,
    "n": 0,
})


/tmp/ipykernel_1740872/513434277.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({


In [17]:
df_long = df.melt(
    id_vars=["uuid", "title", "maintext"],
    value_vars=["jer_relevant", "julia_relevant"],
    var_name="annotator",
    value_name="label"
)

df_matrix = df_long.pivot(
    index="annotator",
    columns="uuid",
    values="label"
)



In [18]:
from simpledorff.simpledorff import calculate_krippendorffs_alpha

alpha = calculate_krippendorffs_alpha(df_matrix)
print(alpha)


0.5485714285714286


In [20]:
df['jer_relevant'].sum()

np.int64(14)

low agreement, 7/40 different. jer_relevant has 14/40 "yes", 0.35. 176/500 = 0.35